# NFL Playoff Predictor

This notebook downloads NFL schedule data, creates team-season features, stores results in SQLite, compares three machine-learning models, and predicts playoff probability. Run the cells in order.

In [32]:
%pip install -q pandas requests scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [33]:
from io import BytesIO
from pathlib import Path
import sqlite3
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Correct nflverse schedule release. It includes historical playoff games.
URL = 'https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv'
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
REPORT_DIR = ROOT / 'reports'
DB_PATH = DATA_DIR / 'nfl_playoffs.sqlite'
NUMERIC = ['games_played', 'wins', 'losses', 'ties', 'win_pct', 'point_diff_per_game', 'points_for_per_game', 'points_against_per_game', 'opponent_win_pct']
CATEGORICAL = ['conference', 'division']
FEATURES = NUMERIC + CATEGORICAL
TEAM_DIVISION = {'ARI':'NFC West','ATL':'NFC South','BAL':'AFC North','BUF':'AFC East','CAR':'NFC South','CHI':'NFC North','CIN':'AFC North','CLE':'AFC North','DAL':'NFC East','DEN':'AFC West','DET':'NFC North','GB':'NFC North','HOU':'AFC South','IND':'AFC South','JAX':'AFC South','KC':'AFC West','LV':'AFC West','LAC':'AFC West','LA':'NFC West','MIA':'AFC East','MIN':'NFC North','NE':'AFC East','NO':'NFC South','NYG':'NFC East','NYJ':'AFC East','PHI':'NFC East','PIT':'AFC North','SF':'NFC West','SEA':'NFC West','TB':'NFC South','TEN':'AFC South','WAS':'NFC East','OAK':'AFC West','SD':'AFC West','STL':'NFC West'}

In [34]:
def load_games():
    response = requests.get(URL, timeout=90)
    response.raise_for_status()
    games = pd.read_csv(BytesIO(response.content), low_memory=False)
    required = {'season', 'game_type', 'home_team', 'away_team', 'home_score', 'away_score'}
    missing = required - set(games.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    games['season'] = pd.to_numeric(games['season'], errors='coerce')
    games['game_type'] = games['game_type'].astype(str).str.strip().str.upper()
    return games.dropna(subset=['season', 'home_team', 'away_team'])

def build_features(games):
    reg = games[(games.game_type == 'REG') & games.home_score.notna() & games.away_score.notna()].copy()
    reg[['home_score', 'away_score']] = reg[['home_score', 'away_score']].apply(pd.to_numeric, errors='coerce')
    home = pd.DataFrame({'season': reg.season, 'team': reg.home_team, 'opponent': reg.away_team, 'pf': reg.home_score, 'pa': reg.away_score})
    away = pd.DataFrame({'season': reg.season, 'team': reg.away_team, 'opponent': reg.home_team, 'pf': reg.away_score, 'pa': reg.home_score})
    long = pd.concat([home, away], ignore_index=True).dropna(subset=['pf', 'pa'])
    long['win'] = (long.pf > long.pa).astype(int)
    long['loss'] = (long.pf < long.pa).astype(int)
    long['tie'] = (long.pf == long.pa).astype(int)
    teams = long.groupby(['season', 'team'], as_index=False).agg(games_played=('pf', 'size'), wins=('win', 'sum'), losses=('loss', 'sum'), ties=('tie', 'sum'), points_for=('pf', 'sum'), points_against=('pa', 'sum'))
    teams['win_pct'] = (teams.wins + 0.5 * teams.ties) / teams.games_played
    teams['point_diff_per_game'] = (teams.points_for - teams.points_against) / teams.games_played
    teams['points_for_per_game'] = teams.points_for / teams.games_played
    teams['points_against_per_game'] = teams.points_against / teams.games_played
    opponent_rates = teams[['season', 'team', 'win_pct']].rename(columns={'team': 'opponent', 'win_pct': 'opponent_win_pct'})
    sos = long.merge(opponent_rates, on=['season', 'opponent'], how='left').groupby(['season', 'team'], as_index=False).opponent_win_pct.mean()
    teams = teams.merge(sos, on=['season', 'team'], how='left')
    teams['division'] = teams.team.map(TEAM_DIVISION).fillna('Unknown')
    teams['conference'] = teams.division.str[:3].where(teams.division.ne('Unknown'), 'Unknown')
    # Everything that is not a regular-season game is treated as playoff/postseason.
    post = games[games.game_type != 'REG'].copy()
    
    labels = pd.concat([
        post[['season', 'home_team']].rename(columns={'home_team': 'team'}),
        post[['season', 'away_team']].rename(columns={'away_team': 'team'})
    ]).drop_duplicates()
    
    labels['made_playoffs'] = 1
    
    teams = teams.merge(
        labels[['season', 'team', 'made_playoffs']],
        on=['season', 'team'],
        how='left'
    )
    
    teams['made_playoffs'] = teams['made_playoffs'].fillna(0).astype(int)
    return teams.drop(columns=['points_for', 'points_against'])

games = load_games()
all_features = build_features(games)
print(f'Downloaded {len(games):,} games; feature table has {len(all_features):,} team-seasons.')
all_features.tail()

Downloaded 7,548 games; feature table has 861 team-seasons.


,season,team,games_played,wins,losses,ties,win_pct,point_diff_per_game,points_for_per_game,points_against_per_game,opponent_win_pct,division,conference,made_playoffs
856,2025,SEA,17,14,3,0,0.823529,11.235294,28.411765,17.176471,0.498270,NFC West,NFC,1
857,2025,SF,17,12,5,0,0.705882,3.882353,25.705882,21.823529,0.498270,NFC West,NFC,1
858,2025,TB,17,8,9,0,0.470588,-1.823529,22.352941,24.176471,0.529412,NFC South,NFC,0
859,2025,TEN,17,3,14,0,0.176471,-11.411765,16.705882,28.117647,0.574394,AFC South,AFC,0
860,2025,WAS,17,5,12,0,0.294118,-5.588235,20.941176,26.529412,0.506920,NFC East,NFC,0


In [35]:
def make_pipeline(model):
    # sparse_output=False fixes the GradientBoosting sparse-matrix error.
    prep = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUMERIC),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL),
    ], sparse_threshold=0)
    return Pipeline([('prep', prep), ('model', model)])

def evaluate(train):
    candidates = {
        'logistic_regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42),
        'random_forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1),
        'gradient_boosting': GradientBoostingClassifier(random_state=42),
    }
    folds = GroupKFold(n_splits=min(5, train.season.nunique()))
    results = []
    for name, estimator in candidates.items():
        scores = []
        for fit_idx, test_idx in folds.split(train, groups=train.season):
            fitted = make_pipeline(estimator).fit(train.iloc[fit_idx][FEATURES], train.iloc[fit_idx].made_playoffs)
            probability = fitted.predict_proba(train.iloc[test_idx][FEATURES])[:, 1]
            actual = train.iloc[test_idx].made_playoffs
            scores.append((roc_auc_score(actual, probability), average_precision_score(actual, probability), accuracy_score(actual, probability >= 0.5)))
        results.append({'model': name, 'cv_roc_auc': round(sum(s[0] for s in scores)/len(scores), 4), 'cv_average_precision': round(sum(s[1] for s in scores)/len(scores), 4), 'cv_accuracy': round(sum(s[2] for s in scores)/len(scores), 4)})
    metrics = pd.DataFrame(results).sort_values('cv_roc_auc', ascending=False)
    winner = metrics.iloc[0].model
    return winner, make_pipeline(candidates[winner]).fit(train[FEATURES], train.made_playoffs), metrics

In [36]:
# Change this to the season you want to estimate.
TARGET_SEASON = all_features.season.max()
train = all_features[(all_features.season < TARGET_SEASON) & all_features.made_playoffs.notna()].copy()
train['made_playoffs'] = train.made_playoffs.astype(int)
predictions = all_features[all_features.season == TARGET_SEASON].copy()
if train.empty or predictions.empty:
    raise ValueError(f'No usable data for {TARGET_SEASON}. Select a season present in the downloaded data.')
best_model, model, metrics = evaluate(train)
predictions['playoff_probability'] = model.predict_proba(predictions[FEATURES])[:, 1]
predictions['predicted_playoff'] = (predictions.playoff_probability >= 0.50).astype(int)
predictions['model'] = best_model
predictions = predictions.sort_values('playoff_probability', ascending=False)
metrics

,model,cv_roc_auc,cv_average_precision,cv_accuracy
0,logistic_regression,0.9791,0.9706,0.9105
1,random_forest,0.9735,0.9604,0.9243
2,gradient_boosting,0.9689,0.9561,0.9220


In [37]:
print("Target season:", TARGET_SEASON)
print("Train shape:", train.shape)
print("Predictions shape:", predictions.shape)

print("\nSeason counts:")
print(all_features.groupby('season').size())

print("\nmade_playoffs counts:")
print(all_features.groupby('season')['made_playoffs'].count())

Target season: 2025
Train shape: (829, 14)
Predictions shape: (32, 17)

Season counts:
season
1999    31
2000    31
2001    31
2002    32
2003    32
2004    32
2005    32
2006    32
2007    32
2008    32
2009    32
2010    32
2011    32
2012    32
2013    32
2014    32
2015    32
2016    32
2017    32
2018    32
2019    32
2020    32
2021    32
2022    32
2023    32
2024    32
2025    32
dtype: int64

made_playoffs counts:
season
1999    31
2000    31
2001    31
2002    32
2003    32
2004    32
2005    32
2006    32
2007    32
2008    32
2009    32
2010    32
2011    32
2012    32
2013    32
2014    32
2015    32
2016    32
2017    32
2018    32
2019    32
2020    32
2021    32
2022    32
2023    32
2024    32
2025    32
Name: made_playoffs, dtype: int64


In [38]:
all_features[['season', 'team', 'made_playoffs']].head()

,season,team,made_playoffs
0,1999,ARI,0
1,1999,ATL,0
2,1999,BAL,0
3,1999,BUF,1
4,1999,CAR,0


In [39]:
all_features['made_playoffs'].value_counts(dropna=False)

made_playoffs
0    525
1    336
Name: count, dtype: int64

In [40]:
DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
with sqlite3.connect(DB_PATH) as connection:
    games.to_sql('games', connection, if_exists='replace', index=False)
    all_features.to_sql('team_season_features', connection, if_exists='replace', index=False)
    metrics.to_sql('model_metrics', connection, if_exists='replace', index=False)
    predictions.to_sql('playoff_predictions', connection, if_exists='replace', index=False)
predictions.to_csv(REPORT_DIR / f'playoff_predictions_{TARGET_SEASON}.csv', index=False)
metrics.to_csv(REPORT_DIR / 'model_metrics.csv', index=False)
print(f'Selected model: {best_model}')
print(f'Saved database: {DB_PATH}')
predictions[['team', 'conference', 'wins', 'losses', 'playoff_probability', 'predicted_playoff']]

Selected model: logistic_regression
Saved database: /Users/bhaskarpariti/nfl_games/data/nfl_playoffs.sqlite


,team,conference,wins,losses,playoff_probability,predicted_playoff
856,SEA,NFC,14,3,0.999938,1
838,DEN,AFC,14,3,0.999938,1
850,NE,AFC,14,3,0.999932,1
843,JAX,AFC,13,4,0.999399,1
857,SF,NFC,12,5,0.997346,1
845,LA,NFC,12,5,0.996503,1
841,HOU,AFC,12,5,0.995373,1
832,BUF,AFC,12,5,0.994575,1
854,PHI,NFC,11,6,0.990505,1
834,CHI,NFC,11,6,0.985635,1
